In [1]:
import json
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

from utils.preprocess import clean_email

In [2]:
# Load dataset
df = pd.read_csv("data/emails.csv")
print("Dataset shape:", df.shape)
print(df["label"].value_counts())
df.head()

Dataset shape: (17536, 2)
label
0    10979
1     6557
Name: count, dtype: int64


,email,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",0
1,the other side of * galicismos * * galicismo *...,0
2,re : equistar deal tickets are you still avail...,0
3,\nHello I am your hot lil horny toy.\n I am...,1
4,software at incredibly low prices ( 86 % lower...,1


In [3]:
# Clean the text
df["clean"] = df["email"].apply(clean_email)
df[["email", "clean"]].head(3)

,email,clean
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",re disc uniformitarianism re sex lang dick hud...
1,the other side of * galicismos * * galicismo *...,the other side of galicismos galicismo is a sp...
2,re : equistar deal tickets are you still avail...,re equistar deal tickets are you still availab...


In [4]:
# Split data (stratified, fixed seed)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
print("Train:", len(X_train), " Test:", len(X_test))

Train: 14028  Test: 3508


In [5]:
# Convert text into numbers - fitted on TRAINING emails only
vectorizer = TfidfVectorizer(stop_words="english", max_features=20000,
                             ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)
print("TF-IDF train matrix:", Xtr.shape)

TF-IDF train matrix: (14028, 20000)


In [6]:
# Train and compare models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
}

results = {}
for name, model in models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }
    print(f"\n{name}  accuracy: {results[name]['accuracy'] * 100:.2f}%")
    print(classification_report(y_test, pred, target_names=["Safe", "Phishing"], digits=4))


Random Forest  accuracy: 97.43%
              precision    recall  f1-score   support

        Safe     0.9870    0.9718    0.9793      2196
    Phishing     0.9539    0.9787    0.9661      1312

    accuracy                         0.9743      3508
   macro avg     0.9705    0.9752    0.9727      3508
weighted avg     0.9747    0.9743    0.9744      3508




Logistic Regression  accuracy: 97.83%
              precision    recall  f1-score   support

        Safe     0.9753    0.9904    0.9828      2196
    Phishing     0.9836    0.9581    0.9707      1312

    accuracy                         0.9783      3508
   macro avg     0.9795    0.9743    0.9767      3508
weighted avg     0.9784    0.9783    0.9783      3508


Multinomial Naive Bayes  accuracy: 97.52%
              precision    recall  f1-score   support

        Safe     0.9705    0.9904    0.9804      2196
    Phishing     0.9834    0.9497    0.9663      1312

    accuracy                         0.9752      3508
   macro avg     0.9770    0.9701    0.9733      3508
weighted avg     0.9754    0.9752    0.9751      3508



In [7]:
# Keep the model with the best F1-score for the app
best = max(results, key=lambda n: results[n]["f1"])
print("Best email model:", best)

joblib.dump(models[best], "models/email_model.pkl")
joblib.dump(vectorizer, "models/vectorizer.pkl")
json.dump({"best": best, "results": results, "train_size": len(X_train), "test_size": len(X_test)},
          open("models/email_metrics.json", "w"), indent=2)
print("Model saved successfully!")

Best email model: Logistic Regression


Model saved successfully!
